In [1]:
!pip install torch torchvision

In [8]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from src.utils import load_cifar10_longtail
import numpy as np
from tqdm import tqdm # Thư viện hiện thanh tiến trình

# 1. Thiết lập thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")

# 2. Tải Model DINOv2
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.eval()

# 3. Transform chuẩn
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- Lớp hỗ trợ để đưa dữ liệu vào DataLoader ---
class CifarLongtailDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img

# 4. Hàm trích xuất đặc trưng tối ưu (Xử lý theo Batch)
def extract_features_fast(X_images, batch_size=32):
    dataset = CifarLongtailDataset(X_images, transform=transform)
    # DataLoader giúp gộp các ảnh thành Batch để xử lý song song
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    feats = []
    # tqdm sẽ tạo ra thanh tiến trình cực đẹp
    with torch.no_grad():
        for batch in tqdm(loader, desc="Đang trích xuất đặc trưng"):
            batch = batch.to(device)
            feat = model(batch)
            feats.append(feat.cpu().numpy())
            
    return np.vstack(feats)

# --- THỰC THI ---

X_longtail, Y_longtail = load_cifar10_longtail('../data/cifar-10-batches-py', imbalance_ratio=0.1)

# Trích xuất với Batch size 64 (Nếu máy có GPU mạnh bạn có thể tăng lên 128)
X_features = extract_features_fast(X_longtail, batch_size=64)

print(f"Xong! Kích thước mảng đặc trưng: {X_features.shape}")

# --- QUAN TRỌNG: Lưu lại để không phải chạy lại lần nữa ---
# Bạn hãy tạo folder 'processed_data' nếu chưa có
os.makedirs('../data/processed', exist_ok=True)
np.save('../data/processed/X_features_cifar_0.1.npy', X_features)
np.save('../data/processed/Y_longtail_cifar_0.1.npy', Y_longtail)
print("Đã lưu đặc trưng vào thư mục data/processed/")

Đang sử dụng thiết bị: cpu


Using cache found in C:\Users\QUANG THINH/.cache\torch\hub\facebookresearch_dinov2_main


Lớp 0: giữ lại 5000 ảnh
Lớp 1: giữ lại 3871 ảnh
Lớp 2: giữ lại 2997 ảnh
Lớp 3: giữ lại 2320 ảnh
Lớp 4: giữ lại 1796 ảnh
Lớp 5: giữ lại 1391 ảnh
Lớp 6: giữ lại 1077 ảnh
Lớp 7: giữ lại 834 ảnh
Lớp 8: giữ lại 645 ảnh
Lớp 9: giữ lại 500 ảnh


Đang trích xuất đặc trưng: 100%|██████████| 320/320 [18:55<00:00,  3.55s/it]


Xong! Kích thước mảng đặc trưng: (20431, 384)
Đã lưu đặc trưng vào thư mục data/processed/


In [ ]:
import gc

# Xóa mảng ảnh gốc vì nó rất nặng
if 'X_longtail' in locals():
    del X_longtail

# Ép Python dọn dẹp rác ngay lập tức
gc.collect()

: 

In [10]:
from src.model import hierarchical_kmeans_resampling
import numpy as np

# Cấu hình tham số cho dữ liệu ảnh thực tế
# Tầng 1: 500 cụm, Tầng 2: 200 cụm, Tầng cuối: 100 cụm
k_list_cifar = [500, 200, 100] 
r_t_list_cifar = [10, 5, 2] # Lấy mẫu thưa dần qua các tầng

print("Đang chạy HK-means trên Features của CIFAR-10...")
# Lưu ý: Hàm trả về centroids, chúng ta cần nhãn (labels) để lấy mẫu
# Bạn hãy cập nhật hàm trong src/model.py để trả về labels nếu chưa có, 
# hoặc dùng KMeans của sklearn để gán nhãn lại dựa trên centroids cuối.

centroids_final = hierarchical_kmeans_resampling(
    X_features, k_list_cifar, T=3, m=5, r_t_list=r_t_list_cifar, num_init=5
)

# Gán nhãn cho toàn bộ tập dữ liệu dựa trên centroids cuối cùng
from sklearn.cluster import KMeans
km_final = KMeans(n_clusters=100, init=centroids_final, n_init=1).fit(X_features)
cluster_labels = km_final.labels_

Đang chạy HK-means trên Features của CIFAR-10...
--- Level 1/3 (k=500, r_t=10) ---


: 